In [ ]:
import __main__
import sys, os
project_root = os.path.abspath("..")  # adjust if notebook is elsewhere
sys.path.insert(0, project_root)
from typing import Dict, List, Literal, Tuple, Optional, Any
import logging

import category_encoders as ce
import matplotlib.pyplot as plt

import numexpr as ne # makes numpy operations faster
import numpy as np
import pandas as pd
import polars as pl
from tqdm import tqdm

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score, mean_absolute_error, root_mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.random_projection import GaussianRandomProjection

from catboost import CatBoostRegressor, CatBoostClassifier

from geo_functions import compute_cyclicity_score, split_dataset_to_linear_and_cyclic, make_windows_from_data

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    # print(torch.cuda.memory_reserved(0) / 1e6, "MB reserved")
    # print(torch.cuda.memory_allocated(0) / 1e6, "MB allocated")

import src.param_config.config_paths as P

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.info("Starting process...")
logging.warning("Something looks off...")
logging.error("Something failed.")


In [ ]:
"another weather (https://www.kaggle.com/datasets/muthuj7/weather-dataset?select=weatherHistory.csv)"

df = pd.read_csv('../public_datasets/2D/tabular/another_weather/weatherHistory.csv')

for col in df.columns:
    if df[col].dtype not in [np.float64, np.float32, np.int64, np.int32]:
        continue
    cycle_score = compute_cyclicity_score(df[col].to_numpy())
    print(f"Cyclicity score of feature {col}: {cycle_score}")


In [ ]:
"cities weather (https://www.kaggle.com/datasets/selfishgene/historical-hourly-weather-data?resource=download)"

weather_folder   = "../public_datasets/2D/tabular/Hourly Weather Data 2012-2017"
files_to_combine = ["temperature.csv", "humidity.csv", "wind_speed.csv", "pressure.csv", "wind_direction.csv"]

dfs = {f.split(".")[0]: pd.read_csv(os.path.join(weather_folder, f)) for f in files_to_combine}

# Extract city names (assumes all files have same columns)
cities     = [col for col in dfs["temperature"].columns if col != "datetime"]
timesteps  = len(dfs["temperature"])
properties = len(files_to_combine)

weather_array = np.zeros((len(cities), timesteps, properties), dtype=float)

# Fill array
for p, prop in enumerate(files_to_combine):
    df_prop = dfs[prop.split(".")[0]]  # remove .csv from name
    for c, city in enumerate(cities):
        weather_array[c, :, p] = df_prop[city].values

for c in range(weather_array.shape[0]):        # cities
    for p in range(weather_array.shape[2]):    # properties
        col = weather_array[c, :, p]
        if np.isnan(col).any():
            mean_val = np.nanmean(col)  # compute mean ignoring NaNs
            col[np.isnan(col)] = mean_val
            weather_array[c, :, p] = col
print("Array shape:", weather_array.shape)
print("NaNs remaining:", np.isnan(weather_array).sum())

cycle_score = compute_cyclicity_score(weather_array[0, :, 0])
print(f"Cyclicity score : {cycle_score}")



In [ ]:
"Pseudo-cyclic synthetic dataset (https://archive.ics.uci.edu/dataset/136/pseudo+periodic+synthetic+time+series)"

data = np.loadtxt('../public_datasets/2D/tabular/pseudo_cyclic/synthetic.data')

for i in range(data.shape[1]):
    cycle_score = compute_cyclicity_score(data[:,i])
    print(f"Cyclicity score of feature {i}: {cycle_score}")

In [ ]:
"Traffic flow dataset (https://archive.ics.uci.edu/dataset/608/traffic+flow+forecasting)"

from scipy.io import loadmat

# Load data
data = loadmat("../public_datasets/2D/tabular/traffic_dataset/traffic_dataset.mat")

def flatten_X(mat_array):
    """
    Convert 1xN MATLAB object array of 36x48 matrices into 2D numeric array
    N rows, 36*48 columns
    """
    flattened = []
    for m in mat_array[0]:
        # Ensure numeric type
        flattened.append(np.array(m, dtype=float).reshape(-1))
    return np.stack(flattened, axis=0)

# Flatten input features
X_train_np = flatten_X(data['tra_X_tr'])
X_test_np  = flatten_X(data['tra_X_te'])

# Outputs: already numeric, just transpose to N x 36
y_train_np = data['tra_Y_tr'].T.astype(float)
y_test_np  = data['tra_Y_te'].T.astype(float)

# Convert to Polars
X_train = pl.DataFrame(X_train_np)
X_test  = pl.DataFrame(X_test_np)
y_train = pl.DataFrame(y_train_np)
y_test  = pl.DataFrame(y_test_np)

print(X_train.shape, y_train.shape)
print(X_train.head())
print(y_train.head())



In [ ]:
"Electric power data (https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption)"

from ucimlrepo import fetch_ucirepo

def get_household_power_consumption(target: str = "Global_active_power") -> Tuple[pl.DataFrame, pl.Series]:
    # 1. Fetch dataset
    ds = fetch_ucirepo(id=235)
    
    # 2. Combine and clean in Pandas to handle mixed types ('?')
    # For this dataset, ds.data.original contains all columns combined
    df_pd = ds.data.original.copy()
    
    # Coerce all columns to numeric except Date and Time
    for col in df_pd.columns:
        if col not in ["Date", "Time"]:
            df_pd[col] = pd.to_numeric(df_pd[col], errors='coerce')
            
    # 3. Convert to Polars safely
    df = pl.from_pandas(df_pd)
    
    # 4. Handle target and nulls
    if target not in df.columns:
        raise ValueError(f"Target '{target}' not found")
        
    df = df.filter(pl.col(target).is_not_null())
    y = df.get_column(target)
    X = df.drop(target).fill_null(0)
    
    return X, y

X, y = get_household_power_consumption()
print(f"X shape: {X.shape}, y shape: {y.shape}")


for col in X.columns:
    if X[col].dtype not in [np.float64, np.float32, np.int64, np.int32]:
        continue
    cycle_score = compute_cyclicity_score(X[col].to_numpy())
    print(f"Cyclicity score of feature {col}: {cycle_score}")




In [ ]:
"longterm weather (https://www.kaggle.com/datasets/alistairking/weather-long-term-time-series-forecasting)"

df = pd.read_csv('../public_datasets/2D/tabular/longterm_weather/longterm_weather.csv')

for col in df.columns:
    if df[col].dtype not in [np.float64, np.float32, np.int64, np.int32]:
        continue
    cycle_score = compute_cyclicity_score(df[col].to_numpy())
    print(f"Cyclicity score of feature {col}: {cycle_score}")


y_cols = ["rain"]
y      = df[y_cols].to_numpy()
X      = df.drop(columns=y_cols, inplace=False)


In [ ]:
# functions

def reparam_gaussian(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    """Gaussian reparameterization."""
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

def reparam_vmf(mu_dir: torch.Tensor, kappa: torch.Tensor) -> torch.Tensor:
    """vMF has no closed form, so approximate vMF sampling: Gaussian noise + projection. Radius = 1 by construction.
    This is what Davidson (2018) does as well, fine because ELBO needs approximate sampling"""
    eps = torch.randn_like(mu_dir)
    z   = mu_dir + eps / (kappa + 1e-6)
    return F.normalize(z, dim= -1) #this projects to unit sphere

class EuclidEncoder(nn.Module):
    def __init__(self, window_size, input_dim, z_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(window_size * input_dim, hidden),
            nn.ReLU())
        self.mu     = nn.Linear(hidden, z_dim)
        self.logvar = nn.Linear(hidden, z_dim)

    def forward(self, x):
        h = self.net(x)
        return self.mu(h), self.logvar(h)

class SphericalEncoder(nn.Module):
    def __init__(self, window_size, input_dim, z_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(window_size * input_dim, hidden),
            nn.ReLU())
        self.mu_raw = nn.Linear(hidden, z_dim)
        self.kappa  = nn.Linear(hidden, 1)

    def forward(self, x):
        h      = self.net(x)
        mu_dir = F.normalize(self.mu_raw(h), dim=-1)
        kappa  = F.softplus(self.kappa(h)) + 1e-3
        return mu_dir, kappa

class Decoder(nn.Module):
    def __init__(self, z_dim_total, window_size, output_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim_total, hidden),
            nn.ReLU(),
            nn.Linear(hidden, window_size * output_dim))
        self.window_size = window_size
        self.output_dim  = output_dim

    def forward(self, z):
        x = self.net(z)
        return x.view(-1, self.window_size, self.output_dim)


def kl_gaussian(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu**2 - logvar.exp(), dim=1).mean()

def regularization_vmf(kappa, dim):
    """Spherical has no closed form of the KL term, so this functino is a proxy regularizer
    This is more of a concentration regularizer encouraging proximity to a uniform hyperspherical prior than a KL divergence."""
    return (kappa - (dim - 1) * torch.log(kappa + 1e-6)).mean()


def vae_train_step(x_lin, x_cyc, x_full, encoder_e, encoder_s, dec, lambdas: dict):
    """Single training step."""
    mu_e, logvar_e = encoder_e(x_lin)
    mu_s, kappa    = encoder_s(x_cyc)

    z_e    = reparam_gaussian(mu_e, logvar_e)
    z_s    = reparam_vmf(mu_s, kappa)
    z      = torch.cat([z_e, z_s], dim=-1)
    x_hat  = dec(z)

    L_reconstr = F.mse_loss(x_hat, x_full)
    L_kl_e     = kl_gaussian(mu_e, logvar_e)
    L_reg_s    = regularization_vmf(kappa, z_s.size(-1))
    loss       = (lambdas["rec"] * L_reconstr + lambdas["euc"] * L_kl_e + lambdas["sph"] * L_reg_s)
    return loss

@torch.no_grad()
def encode_dataset(x_lin, x_cyc, encoder_e, encoder_s):
    mu_e, logvar_e = encoder_e(x_lin)
    mu_s, kappa    = encoder_s(x_cyc)
    z_e = mu_e                      # use mean
    z_s = mu_s                      # already unit norm
    return torch.cat([z_e, z_s], dim=-1)

def train_vae(loader, encoder_e, encoder_s, dec, optimizer, lambdas):
    encoder_e.train(); encoder_s.train(); dec.train()

    for x_lin, x_cyc, x_full in loader:
        optimizer.zero_grad()
        loss = vae_train_step(x_lin, x_cyc, x_full,encoder_e, encoder_s, dec, lambdas)
        loss.backward()
        optimizer.step()



In [ ]:
"Setup"

# train/test split + scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_scaler = StandardScaler()
y_train  = y_scaler.fit_transform(y_train)
y_test   = y_scaler.transform(y_test)

X_scaler = StandardScaler()
X_train  = pd.DataFrame(X_scaler.fit_transform(X_train), columns=X_train.columns)
X_test   = pd.DataFrame(X_scaler.transform(X_test), columns=X_test.columns)

X_lin_train, X_cyc_train = split_dataset_to_linear_and_cyclic(X_train)
X_lin_test, X_cyc_test   = split_dataset_to_linear_and_cyclic(X_test)
X_lin_train = torch.tensor(X_lin_train.values, dtype=torch.float32)
X_cyc_train = torch.tensor(X_cyc_train.values, dtype=torch.float32)
X_lin_test  = torch.tensor(X_lin_test.values, dtype=torch.float32)
X_cyc_test  = torch.tensor(X_cyc_test.values, dtype=torch.float32)

window_size   = 64
X_lin_train_w = make_windows_from_data(X_lin_train, window_size)
X_cyc_train_w = make_windows_from_data(X_cyc_train, window_size)

X_lin_test_w  = make_windows_from_data(X_lin_test, window_size)
X_cyc_test_w  = make_windows_from_data(X_cyc_test, window_size)


z_dim_total = 16
euclid_dim  = z_dim_total // 2
spheric_dim = z_dim_total - euclid_dim
epochs      = 50

encoder_e = EuclidEncoder(window_size=window_size, input_dim=X_lin_train.shape[1], z_dim=euclid_dim, hidden=128).to(device)
encoder_s = SphericalEncoder(window_size=window_size, input_dim=X_cyc_train.shape[1], z_dim=spheric_dim, hidden=128).to(device)
decoder   = Decoder(z_dim_total=euclid_dim + spheric_dim, window_size=window_size, output_dim=X_train.shape[1], hidden=128).to(device)

optimizer = torch.optim.AdamW(list(encoder_e.parameters()) + list(encoder_s.parameters())+ list(decoder.parameters()), lr=1e-3)

train_ds     = TensorDataset(X_lin_train_w, X_cyc_train_w, X_full_train)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

lambdas = {"rec": 1.0, "euc": 1e-3, "sph": 1e-3}

for epoch in range(epochs):
    train_vae(train_loader, encoder_e, encoder_s, decoder, optimizer, lambdas)


Z_train = encode_dataset(X_lin_train_w, X_cyc_train_w, encoder_e, encoder_s)
Z_test  = encode_dataset(X_lin_test_w,  X_cyc_test_w,  encoder_e, encoder_s)

# optional
Z_train = Z_train.mean(dim=0)
Z_test  = Z_test.mean(dim=0)


model   = CatBoostRegressor(verbose=0)
model.fit(Z_train.cpu().numpy(), y_train)
y_hat = model.predict(Z_test.cpu().numpy())


In [ ]:
# step 0: load file
# dataset_loc = "/Users/fouadabiad/Downloads/PAMAP2_Dataset"
dataset_loc = "../public_datasets/2D/tabular/PAMAP2"
record_name = "Protocol/subject108.dat"
record_path = f"{dataset_loc}/{record_name}"
df          = pd.read_csv(record_path, sep=' ', header=None)

# step 1: preprocess
df = df.bfill().ffill()  # fill NaNs for all columns immediately
df = df.sample(frac=0.9, random_state=42)

y  = df.iloc[:, 1].values
df = df.drop(columns=[1]) # drop timestamp and subject ID and activity ID

# scale + split
X_train_np, X_test_np, y_train, y_test = train_test_split(df.values, y, test_size=0.2, random_state=42)
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_np)
X_test_scaled  = scaler.transform(X_test_np)

# 3. Convert to tensors
X_train    = torch.from_numpy(X_train_scaled).float().to(device)
X_test     = torch.from_numpy(X_test_scaled).float().to(device)
y_train_np = np.array(y_train)
y_test_np  = np.array(y_test)

# remove labels (on X_train only)
N   = X_train.shape[0]
rng = torch.randperm(N)
n_l = int(0.2 * N)

idx_L = rng[:n_l]
idx_U = rng[n_l:]

X_L = X_train[idx_L]
y_L = torch.tensor(y_train_np[idx_L.cpu().numpy()], dtype=torch.float32, device=device)
X_U = X_train[idx_U]
y_U = torch.tensor(y_train_np[idx_U.cpu().numpy()], dtype=torch.float32, device=device)  # for eval only


In [ ]:
"VAE vs WAE on labeled/unlabeled sets"
# --- Settings ---
layer_dims      = [128, 64]
latent_dim      = 12
batch_size      = 128
n_clusters      = 13 #12
regressor_type  = "rf"     # "linear", "rf", "catboost"
beta_vae        = 1e-3
lambda_mmd_wae  = 10

input_dim     = X_L.shape[1]
train_dataset = TensorDataset(X_L)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# --- WAE ---
wae_model = WAE(input_dim=input_dim, latent_dim=latent_dim, layer_dims=layer_dims).to(device)
train_wae(wae_model, train_loader, epochs=100, lr=1e-3, lambda_mmd=lambda_mmd_wae, device=device)

with torch.no_grad():
    z_train_wae     = wae_model.encode(X_L)  # labeled
    z_unlabeled_wae = wae_model.encode(X_U)  # unlabeled

# --- VAE ---
vae_model = train_vae(X_L, latent_dim=latent_dim, epochs=100, lr=1e-3,
                      vae_layer_dims=layer_dims, beta=beta_vae)
with torch.no_grad():
    mu_train, logvar_train = vae_model.encode(X_L)
    eps_train   = torch.randn_like(mu_train)
    z_train_vae = mu_train + eps_train * torch.exp(0.5 * logvar_train)

    mu_unlabeled, logvar_unlabeled = vae_model.encode(X_U)
    eps_u           = torch.randn_like(mu_unlabeled)
    z_unlabeled_vae = mu_unlabeled + eps_u * torch.exp(0.5 * logvar_unlabeled)

# --- Evaluate on unlabeled set ---
acc_wae, f1_wae, clusters_wae, sil_score_wae = evaluate_latent(
    z_train_wae, z_unlabeled_wae, y_L.cpu().numpy(), y_U.cpu().numpy(),
    n_clusters=n_clusters, model_type=regressor_type, task_type="classification")

acc_vae, f1_vae, clusters_vae, sil_score_vae = evaluate_latent(
    z_train_vae, z_unlabeled_vae, y_L.cpu().numpy(), y_U.cpu().numpy(),
    n_clusters=n_clusters, model_type=regressor_type, task_type="classification")

# train test sets
with torch.no_grad():
    z_test_wae = wae_model.encode(X_test)
    mu_test, logvar_test = vae_model.encode(X_test)
    eps_test   = torch.randn_like(mu_test)
    z_test_vae = mu_test + eps_test * torch.exp(0.5 * logvar_test)

acc_wae, f1_wae, clusters_wae, sil_test_wae = evaluate_latent(
    z_train_wae, z_test_wae, y_L.cpu().numpy(), y_test_np,
    n_clusters=n_clusters, model_type=regressor_type, task_type="classification")

acc_vae, f1_vae, clusters_vae, sil_test_vae = evaluate_latent(
    z_train_vae, z_test_vae, y_L.cpu().numpy(), y_test_np,
    n_clusters=n_clusters, model_type=regressor_type, task_type="classification")

print("==== Labeled/unlabeled sets ====")
print(f"WAE: Accuracy(↑)={acc_wae:.4f}, F1(↑)={f1_wae:.4f}, Silhouette(↑)={sil_score_wae:.4f}")
print(f"VAE: Accuracy(↑)={acc_vae:.4f}, F1(↑)={f1_vae:.4f}, Silhouette(↑)={sil_score_vae:.4f}")
print("==== Train/test sets ====")
print(f"WAE: Test Accuracy(↑)={acc_wae:.4f}, F1(↑)={f1_wae:.4f}, Silhouette(↑)={sil_test_wae:.4f}")
print(f"VAE: Test Accuracy(↑)={acc_vae:.4f}, F1(↑)={f1_vae:.4f}, Silhouette(↑)={sil_test_vae:.4f}")

with torch.no_grad():
    z_wae = wae_model.encode(X_L)
    z_vae_mu, z_vae_logvar = vae_model.encode(X_L)
    z_vae = z_vae_mu + torch.randn_like(z_vae_mu) * torch.exp(0.5 * z_vae_logvar)

wae_geom = compute_latent_geometry_stats(wae_model.decode, z_wae)
vae_geom = compute_latent_geometry_stats(vae_model.decode, z_vae)

print("WAE geo:", wae_geom)
print("VAE geo:", vae_geom)


In [ ]:
# dimensionality estimation on z
import geomstats.geometry.hypersphere as hs
from geomstats.learning.pca import TangentPCA

latent_dim = 20
vae_model = train_vae(X_train, latent_dim=latent_dim, epochs=100, lr=1e-3,
                      vae_layer_dims=layer_dims, beta=beta_vae)
with torch.no_grad():
    mu_train, logvar_train = vae_model.encode(X_train)
    eps_train   = torch.randn_like(mu_train)
    z_train_vae = mu_train + eps_train * torch.exp(0.5 * logvar_train)
    z_train_vae = z_train_vae.cpu().numpy()

pca = PCA()
pca.fit(z_train_vae)
explained     = np.cumsum(pca.explained_variance_ratio_)
intrinsic_dim = np.searchsorted(explained, 0.9) + 1  # 90% variance
print("Estimated intrinsic dim (PCA):", intrinsic_dim)

def twoNN(X):
    nbrs   = NearestNeighbors(n_neighbors=3).fit(X)
    distances, _ = nbrs.kneighbors(X)
    r2     = distances[:,2] / distances[:,1]  # 2nd NN / 1st NN
    id_est = (np.mean(np.log(r2)))**(-1)
    return id_est

print("Estimated intrinsic dim (TwoNN):", twoNN(z_train_vae))

sphere = hs.Hypersphere(dim=latent_dim)  # e.g., S^k
z_proj = z_train_vae / np.linalg.norm(z_train_vae, axis=1, keepdims=True)  # project onto sphere
pga    = TangentPCA(sphere, n_components=None)
pga.fit(z_proj)
explained     = np.cumsum(pga.explained_variance_ratio_)
intrinsic_dim = np.searchsorted(explained, 0.9) + 1
print("Estimated intrinsic dim (PGA):", intrinsic_dim)


In [ ]:

def plot_latent_tsne(z_wae, z_vae, y=None, perplexity=30, random_state=42):
    """t-SNE visualization of WAE and VAE latent spaces side by side.
    Args:
        z_wae, z_vae: latent representations (torch.Tensor or np.ndarray)
        y: optional labels for coloring
        perplexity: t-SNE perplexity
        random_state: random seed for reproducibility"""
    # ensure numpy arrays
    if isinstance(z_wae, torch.Tensor):
        z_wae = z_wae.cpu().numpy()
    if isinstance(z_vae, torch.Tensor):
        z_vae = z_vae.cpu().numpy()
    
    tsne     = TSNE(n_components=2, perplexity=perplexity, random_state=random_state)
    z_wae_2d = tsne.fit_transform(z_wae)
    z_vae_2d = tsne.fit_transform(z_vae)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].scatter(z_wae_2d[:, 0], z_wae_2d[:, 1], c=y, cmap='tab10', s=15)
    axes[0].set_title("WAE Latent Space (t-SNE)")
    
    axes[1].scatter(z_vae_2d[:, 0], z_vae_2d[:, 1], c=y, cmap='tab10', s=15)
    axes[1].set_title("VAE Latent Space (t-SNE)")
    plt.show()

z_wae_all = torch.cat([z_train_wae, z_unlabeled_wae], dim=0)
z_vae_all = torch.cat([z_train_vae, z_unlabeled_vae], dim=0)
y_all     = np.concatenate([y_L.cpu().numpy(), y_U.cpu().numpy()])

plot_latent_tsne(z_wae_all, z_vae_all, y=y_all)


In [ ]:
"riemannian + VAE"
vae         = train_vae(X_train, latent_dim=8, lr=1e-3, vae_layer_dims=[128, 64], epochs=200, activation=nn.Softplus, distribution="gaussian")
k_prefilter = 30
k_neighbors = 10

visualize_latent_geometry(vae, latent_dim=8) # or 24
# ---------- Encode using mu (deterministic) ----------
with torch.no_grad():
    X_train_t   = X_train.detach().clone().float().to(device)
    X_test_t    = X_test.detach().clone().float().to(device)
    mu_train, _ = vae.encode(X_train_t)
    mu_test, _  = vae.encode(X_test_t)

z_L, z_U = mu_train, mu_test
y_L, y_U = torch.tensor(y_train_np, dtype=torch.float32).to(device), \
           torch.tensor(y_test_np, dtype=torch.float32).to(device)

# ---------- Euclidean prefilter ----------
d2_eucl  = torch.cdist(z_U, z_L)**2
idx_pref = torch.topk(d2_eucl, k_prefilter, largest=False).indices

# ---------- Cache G ----------
G_L = torch.stack([compute_G(z, vae.decode) for z in z_L])
G_U = torch.stack([compute_G(z, vae.decode) for z in z_U])

# ---------- Midpoint Riemannian distances ----------
Nu      = z_U.shape[0]
d2_riem = torch.full_like(d2_eucl, float('inf'))

for i in range(Nu):
    z_l_sel = z_L[idx_pref[i]]      # (k_prefilter, d)
    G_l_sel = G_L[idx_pref[i]]      # (k_prefilter, d, d)
    dz      = z_U[i] - z_l_sel      # (k_prefilter, d)
    G_mid   = 0.5 * (G_U[i] + G_l_sel)  # (k_prefilter, d, d)
    d2      = torch.einsum('kd,kdd,kd->k', dz, G_mid, dz)
    d2_riem[i, idx_pref[i]] = d2

# ---------- kNN regression ----------
y_pred_euclid  = knn_regress(d2_eucl, y_L, k=k_neighbors)
y_pred_riemann = knn_regress(d2_riem, y_L, k=k_neighbors)

# ---------- Eval ----------
rmse   = lambda a, b: torch.sqrt(((a - b) ** 2).mean())
print(f"RMSE Euclidean: {rmse(y_pred_euclid, y_U).item():.3f}")
print(f"RMSE Riemannian: {rmse(y_pred_riemann, y_U).item():.3f}")
mean_G = G_L.mean()
max_G  = G_L.max()
print(f"Mean metric G(z): {mean_G:.4f}")
print(f"Max metric G(z):  {max_G:.4f}")

print("=== Local geometry at 1 latent point ===")
detG    = torch.det(G_L[0]) # local volume distortion
eigvals = torch.linalg.eigvalsh(G_L[0]) # if ≈ constant -> flat, euclidean distance is enough, manifold geometry adds nothing
print(f"det(G): {detG.item():.3f}")
print("eigenvalues:", eigvals)


In [ ]:
"riemanian + AE"

# ---------- Init ----------
x_dim       = X_L.shape[1]
z_dim       = 2
k_prefilter = 25**2  # Euclidean prefilter
k_neighbors = 5   # num of kNN neighbors
epochs      = 100
lr_optim    = 1e-3

encoder   = Encoder(x_dim, z_dim).to(device)
decoder   = Decoder(z_dim, x_dim).to(device)
optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(decoder.parameters()), lr=lr_optim)

# ---------- Train simple AE ----------
for _ in range(epochs):
    optimizer.zero_grad()
    X_all = torch.cat([X_L, X_U], dim=0)
    z_all = encoder(X_all)
    x_hat = decoder(z_all)
    loss  = ((X_all - x_hat) ** 2).mean()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    z_L = encoder(X_L)
    z_U = encoder(X_U)

# ---------- Euclidean prefilter ----------
d2_euclid = torch.cdist(z_U, z_L) ** 2
idx_pref  = torch.topk(d2_euclid, k_prefilter, largest=False).indices  # (Nu, k_prefilter)

G_L = torch.stack([compute_G(z, decoder) for z in z_L])
G_U = torch.stack([compute_G(z, decoder) for z in z_U])

# ---------- Riemannian distances (midpoint + prefilter) ----------
num_U      = z_U.shape[0]
d2_riemann = torch.full_like(d2_euclid, float('inf'))  # init
for i in range(num_U):
    z_l_sel = z_L[idx_pref[i]]        # (k_prefilter, d)
    G_l_sel = G_L[idx_pref[i]]        # (k_prefilter, d, d)
    dz      = z_U[i] - z_l_sel          # (k_prefilter, d)
    G_mid   = 0.5 * (G_U[i] + G_l_sel)  # (k_prefilter, d, d)
    d2      = torch.einsum('kd,kdd,kd->k', dz, G_mid, dz)  # (k_prefilter,)
    d2_riemann[i, idx_pref[i]] = d2

# ---------- kNN regression ----------
y_pred_euclid  = knn_regress(d2_euclid, y_L, k=k_neighbors)
y_pred_riemann = knn_regress(d2_riemann, y_L, k=k_neighbors)
# ---------- Eval ----------
rmse   = lambda a, b: torch.sqrt(((a - b) ** 2).mean())
print(f"RMSE Euclidean: {rmse(y_pred_euclid, y_U).item():.3f}")
print(f"RMSE Riemannian: {rmse(y_pred_riemann, y_U).item():.3f}")
mean_G = G_L.mean()
max_G  = G_L.max()
print(f"Mean metric G(z): {mean_G:.4f}")
print(f"Max metric G(z):  {max_G:.4f}")

print("=== Local geometry at 1 latent point ===")
detG    = torch.det(G_L[0]) # local volume distortion
eigvals = torch.linalg.eigvalsh(G_L[0]) # if ≈ constant -> flat, euclidean distance is enough, manifold geometry adds nothing
print(f"det(G): {detG.item():.3f}")
print("eigenvalues:", eigvals)


In [ ]:
"visualize folding/stretching/holes"
# z in R^2 → x in R^2
grid = torch.stack(torch.meshgrid(
    torch.linspace(-3, 3, 200),
    torch.linspace(-3, 3, 200)), dim=-1)

x = decoder(grid.reshape(-1, 2))
plt.plot(x.detach().numpy())

print(x.shape, grid.shape)


In [ ]:
"umap"
import umap
from sklearn.metrics import accuracy_score, r2_score

# ===== 1. Fit UMAP embedding =====
n_neighbors  = 11
n_components = 5  # same as your Isomap embedding
umap_model   = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.1, n_components=n_components, random_state=42)
X_train_umap = umap_model.fit_transform(X_train_scaled)
X_test_umap  = umap_model.transform(X_test_scaled)

# ===== 2. Plot first 3 components for visualization =====
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    X_train_umap[:,0],
    X_train_umap[:,1],
    X_train_umap[:,2],
    c=y_train_np,
    cmap='tab10',
    alpha=0.8)
fig.colorbar(scatter, label='Class/Label')
ax.set_title("UMAP 3D Embedding")
plt.show()

# ===== 3. RandomForestClassifier on UMAP embedding =====
rf_classifier= RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_umap, y_train_np)
y_pred_class = rf_classifier.predict(X_test_umap)
accuracy     = accuracy_score(y_test_np, y_pred_class)
rmse_class   = root_mean_squared_error(y_test_np, y_pred_class)
print(f"UMAP accuracy (classif.): {accuracy:.3f}")
print(f"UMAP RMSE (classif.): {rmse_class:.3f}")

# ===== 4. RandomForestRegressor on UMAP embedding =====
rf_regressor= RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_umap, y_train_np)
y_pred_reg  = rf_regressor.predict(X_test_umap)
r2          = r2_score(y_test_np, y_pred_reg)
rmse_reg    = root_mean_squared_error(y_test_np, y_pred_reg)
print(f"UMAP R² (regression): {r2:.3f}")
print(f"UMAP RMSE (regression): {rmse_reg:.3f}")


In [ ]:
"kernel PCA"
from sklearn.decomposition import KernelPCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

# ===== 1. Sweep embedding dimension to estimate residual variance (optional) =====
# Note: KernelPCA does not give a built-in dist_matrix_, so residual variance calculation is less straightforward.
# Here we can skip it or just look at explained variance if using linear kernel.
# For nonlinear kernels, you typically pick n_components based on domain knowledge or grid search.

# ===== 2. Fit kernel PCA with chosen embedding dimension =====
n_components = 8  # same as your Isomap example
kpca_model   = KernelPCA(n_components=n_components, kernel='rbf', gamma=0.05, fit_inverse_transform=True)
X_train_kpca = kpca_model.fit_transform(X_train)
X_test_kpca  = kpca_model.transform(X_test)

# ===== 3. Plot first 3 components (for visualization) =====
fig = plt.figure(figsize=(8,6))
ax  = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    X_train_kpca[:,0],
    X_train_kpca[:,1],
    X_train_kpca[:,2],
    c=y_train_np,
    cmap='tab10',
    alpha=0.8)
fig.colorbar(scatter, label='Class/Label')
ax.set_title("Kernel PCA 3D Embedding")
plt.show()

# ===== 4. RandomForestClassifier on KPCA embedding =====
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_kpca, y_train_np)
y_pred_class = rf_classifier.predict(X_test_kpca)
accuracy     = accuracy_score(y_test_np, y_pred_class)
rmse_class   = np.sqrt(mean_squared_error(y_test_np, y_pred_class))
print(f"Kernel PCA accuracy (classif.): {accuracy:.3f}")
print(f"Kernel PCA RMSE (classif.): {rmse_class:.3f}")

# ===== 5. RandomForestRegressor on KPCA embedding =====
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_kpca, y_train_np)
y_pred_reg = rf_regressor.predict(X_test_kpca)
r2        = r2_score(y_test_np, y_pred_reg)
rmse_reg  = np.sqrt(mean_squared_error(y_test_np, y_pred_reg))
print(f"Kernel PCA R² (regression): {r2:.3f}")
print(f"Kernel PCA RMSE (regression): {rmse_reg:.3f}")


In [ ]:
"isomap"
from sklearn.manifold import Isomap
import warnings
from scipy.sparse import SparseEfficiencyWarning
warnings.simplefilter('ignore', SparseEfficiencyWarning)

# ===== 1. plot residual variance to estimate intrinsic dimensionality (finds the best dim)
res_vars       = []
embedding_dims = range(1, 11)  # try 1D to 10D embeddings
for dim in embedding_dims:
    iso = Isomap(n_neighbors=20, n_components=dim)
    iso.fit(X_train)
    # residual variance = 1 - R^2 between graph distances and embedding distances
    dist_graph = iso.dist_matrix_
    dist_emb   = np.linalg.norm(iso.embedding_[:, None, :] - iso.embedding_[None, :, :], axis=2)
    r2         = np.corrcoef(dist_graph.ravel(), dist_emb.ravel())[0,1]**2
    res_vars.append(1 - r2)
plt.plot(embedding_dims, res_vars, marker='o')
plt.xlabel("Embedding dimension")
plt.ylabel("Residual variance")
plt.title("Estimate intrinsic dimensionality")
plt.show()

# ===== 2. plot isomap of training data (uses the best estimated dim from above)
isomap_model   = Isomap(n_neighbors=20, n_components=5)
X_train_isomap = isomap_model.fit_transform(X_train)
X_test_isomap  = isomap_model.transform(X_test)

fig     = plt.figure(figsize=(8,6))
ax      = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    X_train_isomap[:,0],
    X_train_isomap[:,1],
    X_train_isomap[:,2],
    c=y_train_np,
    cmap='tab10',
    alpha=0.8)
fig.colorbar(scatter, label='Class/Label')  # attach to figure
ax.set_title("Isomap 3D Embedding")
plt.show()


In [ ]:
"isomap results"
rf_classifier= RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_isomap, y_train_np)          # train on Isomap embedding
y_pred_class = rf_classifier.predict(X_test_isomap)          # predict test labels
accuracy     = accuracy_score(y_test_np, y_pred_class)
print(f"Isomap accuracy (classif.): {accuracy:.3f}")
rmse         = np.sqrt(mean_squared_error(y_test_np, y_pred_class))
print(f"Isomap RMSE (classif.): {rmse:.3f}")

# ====
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_isomap, y_train_np)           # train on Isomap embedding
y_pred_reg   = rf_regressor.predict(X_test_isomap)      # predict on test embedding
r2           = r2_score(y_test_np, y_pred_reg)
print(f"Isomap R² (regression): {r2:.3f}")
test_rmse    = np.sqrt(mean_squared_error(y_test_np, y_pred_reg))
print(f"Isomap RMSE (regression): {test_rmse:.3f}")


In [ ]:
# isomap (above)
# ===============
# riemann (below)

In [ ]:
"""FULL MANIFOLD-AWARE VAE CLUSTERING CELL"""
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import SpectralClustering
import networkx as nx

class VAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, layer_dims: list = [128, 64], distribution: str = "gaussian"):
        super().__init__()
        self.distribution = distribution
        # encoder
        enc_layers = []
        prev       = input_dim
        for h in layer_dims:
            enc_layers.append(nn.Linear(prev, h))
            enc_layers.append(nn.ReLU())
            prev = h
        self.enc    = nn.Sequential(*enc_layers)
        self.mu     = nn.Linear(prev, latent_dim)
        self.logvar = nn.Linear(prev, latent_dim)

        # Decoder (mirror encoder)
        dec_layers = []
        prev       = latent_dim
        for h in reversed(layer_dims):
            dec_layers.append(nn.Linear(prev, h))
            dec_layers.append(nn.ReLU())
            prev = h
        dec_layers.append(nn.Linear(prev, input_dim))
        self.dec = nn.Sequential(*dec_layers)

    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        """for gaussian or von Mises-Fisher"""
        if self.distribution.lower()[0] == "g": # gaussian
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        elif self.distribution.lower()[0] == "v": # von Mises-Fisher
            mu    = F.normalize(mu, dim=1)
            kappa = F.softplus(logvar) + 1e-6
            eps   = torch.randn_like(mu)
            eps   = F.normalize(eps, dim=1)
            return F.normalize(mu + eps / kappa, dim=1)
        elif self.distribution.lower()[0] == "s": # pure spherical 
            return F.normalize(mu, dim=1)

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z          = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar if self.distribution.lower()[0] != "s" else z


def vae_loss(x, x_hat, mu, logvar, distribution: str = "gaussian"):
    "depends on distr., Gaussian or von Mises-Fisher"
    recon = F.mse_loss(x_hat, x, reduction="mean")
    if distribution.lower()[0] == "g": # gaussian
        # KL divergence between q(z|x) and p(z) = N(0, I)
        kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return recon + kl
    elif distribution.lower()[0] == "v": # von Mises-Fisher
        # normalize mu to lie on unit sphere
        mu    = F.normalize(mu, dim=1)
        kappa = F.softplus(logvar) + 1e-6
        kl    = torch.mean(kappa) # KL to uniform-on-sphere (approx)
        return recon + 0.1 * kl
    elif distribution.lower()[0] == "s": # pure spherical
        return recon 

def train_vae(X_train, latent_dim=8, lr=1e-3, epochs=100, distribution="gaussian"):
    input_dim = X_train.shape[1]
    vae       = VAE(input_dim, latent_dim,layer_dims=[32, 16], distribution=distribution).to(device)
    optimizer = torch.optim.AdamW(vae.parameters(), lr=lr)
    # X_train_t = torch.tensor(X_train.values, dtype=torch.float32).to(device)
    # X_train_t = torch.tensor(X_train if isinstance(X_train, np.ndarray) else X_train.values,
    #                      dtype=torch.float32).to(device)
    if isinstance(X_train, np.ndarray):
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    else:
        # pandas DataFrame
        X_train_t = torch.tensor(np.array(X_train), dtype=torch.float32).to(device)


    for epoch in range(epochs):
        optimizer.zero_grad()
        x_hat, mu, logvar = vae(X_train_t)
        loss              = vae_loss(X_train_t, x_hat, mu, logvar, distribution=distribution)
        loss.backward()
        optimizer.step()
        if epoch % 5 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
    return vae

def compute_latent_graph(vae, X, knn_neighbors=30):
    print("[Graph] Encoding latent variables...")
    X_input = np.array(X) if not isinstance(X, np.ndarray) else X
    with torch.no_grad():
        Z = vae.encode(torch.tensor(X_input, dtype=torch.float32).to(device))[0]

    Z_np = Z.cpu().numpy()
    n    = Z_np.shape[0]
    print(f"[Graph] Latent shape: {Z_np.shape}")

    # optional: compute local decoder metric
    def compute_metric(decoder, z):
        z     = z.clone().detach().requires_grad_(True)
        x     = decoder(z)
        grads = [torch.autograd.grad(x[0, i], z, retain_graph=True)[0].squeeze(0) for i in range(x.shape[1])]
        J     = torch.stack(grads, dim=0)
        # print("[Graph] Metrics computed")
        return J.T @ J

    Ms = [compute_metric(vae.dec, Z[i:i+1]) for i in range(n)]
    Ms = torch.stack(Ms)

    # kNN + Riemannian weights
    print("[Graph] Building kNN graph...")
    nbrs      = NearestNeighbors(n_neighbors=knn_neighbors+1).fit(Z_np)
    neighbors = nbrs.kneighbors(Z_np, return_distance=False)

    G = nx.Graph()
    for i in range(n):
        if i % 100 == 0:
            print(f"[Graph] Metric {i}/{n}")
        G.add_node(i)
        zi, Mi = Z[i], Ms[i]
        for j in neighbors[i][1:]:
            dz = zi - Z[j]
            w  = torch.sqrt(dz @ Mi @ dz)
            G.add_edge(i, j, weight=float(w))

    # all-pairs geodesic distances (Dijkstra)
    print("[Graph] Computing geodesic distances (Dijkstra)...")
    geo   = dict(nx.all_pairs_dijkstra_path_length(G))
    D_geo = np.zeros((n, n))
    for i in range(n):
        for j, d_ij in geo[i].items():
            D_geo[i, j] = d_ij
    print("[Graph] Geodesic distances computed")
    return Z_np, D_geo

def manifold_clustering(D_geo, n_clusters=5):
    labels = SpectralClustering(n_clusters=n_clusters, affinity="precomputed", random_state=0)\
        .fit_predict(np.exp(-D_geo))
    return labels

# DATA
input_dim     = X_train.shape[1]
latent_dim    = 10
lr_optim      = 1e-3
n_spectral_clusters = 12
knn_neighbors = 30
epochs        = 100
vae                  = train_vae(X_train, latent_dim=latent_dim, epochs=epochs, distribution="gaussian")
# Z_train, D_geo_train = compute_latent_graph(vae, X_train, knn_neighbors=knn_neighbors)
# labels_train         = manifold_clustering(D_geo_train, n_clusters=5)


In [ ]:
"isomap/umap on z"
# ---- 1. Encode data with VAE ----
with torch.no_grad():
    Z_train = vae.encode(X_train)[0].cpu().numpy()  # shape [n_train, latent_dim]
    Z_test  = vae.encode(X_test)[0].cpu().numpy()

# ---- 2a. Isomap on latent codes ----
embedding_choice = "umap"  # or "umap"
if embedding_choice == "isomap":
    isomap_model= Isomap(n_neighbors=20, n_components=5)
    Z_train_iso = isomap_model.fit_transform(Z_train)
    Z_test_iso  = isomap_model.transform(Z_test)
    X_train_embedded = Z_train_iso
    X_test_embedded  = Z_test_iso
elif embedding_choice == "umap":
    umap_model   = umap.UMAP(n_neighbors=13, min_dist=0.1, n_components=5, random_state=42)
    Z_train_umap = umap_model.fit_transform(Z_train)
    Z_test_umap  = umap_model.transform(Z_test)
    X_train_embedded = Z_train_umap   # or Z_train_iso
    X_test_embedded  = Z_test_umap    # or Z_test_iso

# ---- 3. Train RandomForest on latent embedding ----
rf     = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_embedded, y_train_np)
y_pred = rf.predict(X_test_embedded)
rmse   = root_mean_squared_error(y_test_np, y_pred)
print(f"RMSE for {embedding_choice} on latent manifold embedding: {rmse:.4f}")


In [ ]:
"full riemann clustering results"
from sklearn.metrics import pairwise_distances_argmin_min
import collections

# ---- Convert train latent codes and labels to numpy ----
Z_train_np = Z_train
y_train_np = np.array(y_train) if not isinstance(y_train, np.ndarray) else y_train
y_test_np  = np.array(y_test)  if not isinstance(y_test, np.ndarray)  else y_test

with torch.no_grad():
    Z_test = vae.encode(X_test)[0].cpu().numpy()  # X_test is tensor

# ---- Global RMSE on test ----
reg_global  = RandomForestRegressor(n_estimators=100, random_state=42)
reg_global.fit(Z_train_np, y_train_np)          # train on train latent codes
y_test_pred = reg_global.predict(Z_test)        # predict on test latent codes
test_rmse   = np.sqrt(mean_squared_error(y_test_np, y_test_pred))
print("Test normal RMSE (z -> y):", test_rmse)

#  simple clustering on test
# kNN on xtrain+xtest, then get the avg of the ytrain of that cluster, and this becomes the y_test of points in that clsuter
kmeans       = KMeans(n_clusters=knn_neighbors, random_state=42)
train_labels = kmeans.fit_predict(X_train_np)
y_test_pred_cluster   = np.zeros_like(y_test_np, dtype=float)
rmse_per_cluster_test = []
for c in np.unique(train_labels):
    # get train points in this cluster
    train_idx = train_labels == c
    if np.sum(train_idx) == 0:
        continue
    # train RF only on this cluster
    reg = RandomForestRegressor(n_estimators=100, random_state=42)
    reg.fit(X_train_np[train_idx], y_train_np[train_idx])
    # assign test points nearest to this cluster center
    distances = np.linalg.norm(X_test_np - kmeans.cluster_centers_[c], axis=1)
    # simple nearest-point assignment
    test_idx = distances == distances.min()
    y_test_pred_cluster[test_idx] = reg.predict(X_test_np[test_idx])
    # compute cluster RMSE
    cluster_rmse = np.sqrt(mean_squared_error(y_test_np[test_idx], y_test_pred_cluster[test_idx]))
    rmse_per_cluster_test.append(cluster_rmse)
    print(f"Cluster {c}: RMSE = {cluster_rmse:.4f}")

overall_rmse_test = np.sqrt(mean_squared_error(y_test_np, y_test_pred_cluster))
print("RMSE for simple clustering:", overall_rmse_test)
print("Train cluster counts:", collections.Counter(train_labels))


# ---- Cluster-wise RMSE on test ----
nearest_train_idx, _  = pairwise_distances_argmin_min(Z_test, Z_train_np)
test_labels           = labels_train[nearest_train_idx]
y_test_pred_cluster   = np.zeros_like(y_test_np, dtype=float)
rmse_per_cluster_test = []

for c in np.unique(labels_train):
    idx       = test_labels == c
    train_idx = labels_train == c
    if np.sum(train_idx) == 0 or np.sum(idx) == 0:
        continue  # skip empty clusters
    reg = RandomForestRegressor(n_estimators=100, random_state=42)
    reg.fit(Z_train_np[train_idx], y_train_np[train_idx])
    y_test_pred_cluster[idx] = reg.predict(Z_test[idx])
    cluster_rmse             = np.sqrt(mean_squared_error(y_test_np[idx], y_test_pred_cluster[idx]))
    rmse_per_cluster_test.append(cluster_rmse)
    print(f"Test Cluster {c}: RMSE = {cluster_rmse:.4f}")

overall_rmse_test = np.sqrt(mean_squared_error(y_test_np, y_test_pred_cluster))
print("Test Geometric RMSE (z -> y):", overall_rmse_test)

# Cluster counts in train set
print(collections.Counter(labels_train))


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ---- Use PCA or t-SNE ----
# Z_2d = PCA(n_components=2).fit_transform(Z_train)  # or 
Z_2d = TSNE(n_components=2).fit_transform(Z_train)

# ---- Plot by cluster ----
plt.figure(figsize=(8,6))
for c in np.unique(labels_train):
    idx = labels_train == c
    plt.scatter(Z_2d[idx, 0], Z_2d[idx, 1], label=f"Cluster {c}", alpha=0.6)
plt.legend()
plt.title("Latent space (PCA) colored by cluster")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# ---- Optional: plot by target y ----
plt.figure(figsize=(8,6))
plt.scatter(Z_2d[:, 0], Z_2d[:, 1], c=y_train_np, cmap="viridis", alpha=0.6)
plt.colorbar(label="y value")
plt.title("Latent space colored by target")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


In [ ]:
# riemann (above)
# =======================
# spherical (below)

In [ ]:
# step 1: split features
row = df.values  # shape (num_rows, num_features)

# 1️⃣ Sphere block: 3D vectors (accel + gyro)
sphere_cols = [
    slice(1,4),  slice(7,10),   # hand
    slice(20,23), slice(27,30), # chest
    slice(37,40), slice(44,47)]  # ankle
sphere_vectors  = [row[:, s] for s in sphere_cols]
Z_sphere        = np.concatenate(sphere_vectors, axis=1)  # shape (num_rows, 18)
Z_sphere_scaled = StandardScaler().fit_transform(Z_sphere)

# 2️⃣ Torus block: cyclic/stride phase (mocked here as random)
Z_torus         = np.random.rand(row.shape[0], 1) * 2*np.pi  # shape (num_rows, 1)
Z_torus_scaled  = (Z_torus - np.pi) / np.pi  # scale to [-1,1]

# 3️⃣ Euclidean block: scalar features
euclid_cols     = [2, 3, 20, 37]  # Python indexing
Z_euclid        = row[:, euclid_cols]  # shape (num_rows, 4)
Z_euclid_scaled = StandardScaler().fit_transform(Z_euclid)

# Combine all features into one Euclidean input
Z_all = np.concatenate([Z_sphere_scaled, Z_euclid_scaled], axis=1) # for full Euclidean set

print("Sphere block shape:", Z_sphere.shape)
# print("Torus block shape:", Z_torus.shape)
print("Euclidean block shape:", Z_euclid.shape)
print("ALL shape:", Z_all.shape)


In [ ]:
"(V)AE"
# 1️⃣ Euclidean Autoencoder
class EuclideanAE(nn.Module):
    def __init__(self, input_dim:int, hidden_dim:int, latent_dim:int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim))
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim))
    def forward(self, x:torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return z, x_hat

# 2️⃣ Spherical Autoencoder (vMF prior); uses L2 norm to project to hypersphere
class SphericalAE(nn.Module):
    def __init__(self, input_dim:int, hidden_dim:int, latent_dim:int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim))

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim))
    def forward(self, x:torch.Tensor):
        z_raw = self.encoder(x)
        norms = torch.norm(z_raw, p=2, dim=1, keepdim=True) + 1e-8
        z     = z_raw / norms
        x_hat = self.decoder(z)
        return z, x_hat

# 3️⃣ Torus Autoencoder (wrap-aware); encode phase as sin/cos, decode same
class TorusAE(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),)   # ⬅️ k angles
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),)

    def forward(self, x):
        theta = self.encoder(x)                  # (N, k)
        x_hat = self.decoder(theta)
        return theta, x_hat

def train_ae(
    model: nn.Module,
    x: torch.Tensor,
    epochs: int = 50,
    lr: float = 1e-3,
    reg_fn=None,
    reg_weight: float = 0.0,):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        optimizer.zero_grad()
        z, x_hat = model(x)
        loss     = F.mse_loss(x_hat, x)
        if reg_fn is not None:
            loss = loss + reg_weight * reg_fn(z)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if epoch % 10 == 0 or epoch == epochs-1:
            print(f"Epoch {epoch+1}/{epochs}, loss: {loss.item():.6f}")
    return model


In [ ]:
# step 2: encode to z
x_sphere = torch.tensor(Z_sphere_scaled, dtype=torch.float32)  # (num_rows, 18)
x_torus  = torch.tensor(Z_torus_scaled, dtype=torch.float32)   # (num_rows, 1)
x_euclid = torch.tensor(Z_euclid_scaled, dtype=torch.float32) # (num_rows, 4)
x_all    = torch.tensor(Z_all, dtype=torch.float32) # for full Euclidean set

# --- instantiate encoders ---
sphere_input      = Z_sphere.shape[1]
sphere_latent_dim = 3
sphere_hidden_dim = 16
sphere_ae         = SphericalAE(sphere_input, sphere_hidden_dim, sphere_latent_dim)

torus_input       = Z_torus.shape[1]
latent_dim_torus  = 2
torus_hidden_dim  = 8
torus_ae          = TorusAE(torus_input, torus_hidden_dim, latent_dim_torus)

euclid_input      = Z_euclid.shape[1]
euclid_latent_dim = 2
euclid_hidden_dim = 8
euclid_ae         = EuclideanAE(euclid_input, euclid_hidden_dim, euclid_latent_dim)

# full euclidean set
input_dim  = Z_all.shape[1]
hidden_dim = 16
latent_dim = sphere_latent_dim + euclid_latent_dim
euclid_full_ae = EuclideanAE(input_dim, hidden_dim, latent_dim)

# --- train ---
epochs         = 100
euclid_ae      = train_ae(euclid_ae, x_euclid, epochs=epochs)
sphere_ae      = train_ae(sphere_ae, x_sphere, epochs=epochs)
euclid_full_ae = train_ae(euclid_full_ae, x_all, epochs=epochs)

def torus_uniform_regularizer(theta):
    return (theta.cos().mean()**2 + theta.sin().mean()**2)

torus_ae = train_ae(
    torus_ae,
    x_torus,
    epochs=epochs,
    reg_fn=torus_uniform_regularizer,
    reg_weight=0.1,)

# --- encode ---
z_sphere_enc, _ = sphere_ae(x_sphere)
z_torus_enc,  _ = torus_ae(x_torus)
z_euclid_enc, _ = euclid_ae(x_euclid)

z_euclid_full_enc, _ = euclid_full_ae(x_all)
z_euclid_full_np     = z_euclid_full_enc.detach().numpy()
print("Full Euclidean AE latent shape:", z_euclid_full_np.shape)

# --- concatenate into final hybrid latent vectors ---
# z_hybrid_encoded    = torch.cat([z_sphere_enc, z_torus_enc, z_euclid_enc], dim=1)  # (num_rows, total_latent_dim)
z_hybrid_encoded    = torch.cat([z_sphere_enc, z_euclid_enc], dim=1)  # (num_rows, total_latent_dim)
z_hybrid_encoded_np = z_hybrid_encoded.detach().numpy()
print("Encoded hybrid latent vectors shape:", z_hybrid_encoded.shape)

print("Sphere latent NaNs:", np.isnan(z_sphere_enc.detach().numpy()).sum())
print("Torus latent NaNs:", np.isnan(z_torus_enc.detach().numpy()).sum())
print("Euclid latent NaNs:", np.isnan(z_euclid_enc.detach().numpy()).sum())


In [ ]:
"plot"
z_s = z_sphere_enc.detach().numpy()  # (num_rows, 3)
fig = plt.figure()
ax  = fig.add_subplot(111, projection='3d')
ax.scatter(z_s[:,0], z_s[:,1], z_s[:,2], c='blue', s=5)
ax.set_title("Sphere latent space")
plt.show()

z_t = z_torus_enc.detach().numpy()  # (num_rows, 2)
plt.figure()
plt.scatter(np.cos(z_t[:,0]), np.sin(z_t[:,0]), c='red', s=5, label='theta1')
plt.scatter(np.cos(z_t[:,1]), np.sin(z_t[:,1]), c='green', s=5, label='theta2')
plt.axis('equal')
plt.title("Torus latent space")
plt.legend()
plt.show()

z_e = z_euclid_enc.detach().numpy()  # (num_rows, 2)
plt.scatter(z_e[:,0], z_e[:,1], c='purple', s=5)
plt.title("Euclidean latent space")
plt.show()

Z_hybrid = z_hybrid_encoded_np  # (num_rows, total_latent_dim)
Z_2d = PCA(n_components=2).fit_transform(Z_hybrid)
plt.scatter(Z_2d[:,0], Z_2d[:,1], s=5)
plt.title("Hybrid latent space (PCA 2D)")
plt.show()


In [ ]:
# step 3: hybrid distance (sphere + euclidean)
latent_dim_sphere = z_sphere_enc.shape[1]
latent_dim_euclid = z_euclid_enc.shape[1]

idx_sphere = slice(0, latent_dim_sphere)
idx_euclid = slice(latent_dim_sphere, None)

Z = z_hybrid_encoded_np
num_rows = Z.shape[0]

dist_matrix = np.zeros((num_rows, num_rows))

for i in range(num_rows):
    u   = Z[i]
    u_s = u[idx_sphere]
    u_s = u_s / np.linalg.norm(u_s)

    for j in range(i, num_rows):
        v = Z[j]

        # sphere
        v_s = v[idx_sphere]
        v_s = v_s / np.linalg.norm(v_s)
        d_sphere = np.arccos(np.clip(np.dot(u_s, v_s), -1.0, 1.0))

        # euclidean
        u_e, v_e = u[idx_euclid], v[idx_euclid]
        d_euclid = np.linalg.norm(u_e - v_e)

        d = d_sphere + d_euclid
        dist_matrix[i, j] = d
        dist_matrix[j, i] = d


In [ ]:
# torus check
theta = z_torus_enc.detach().cpu().numpy().ravel()

print("mean cos:", np.mean(np.cos(theta)))
print("mean sin:", np.mean(np.sin(theta)))
plt.hist(theta % (2*np.pi), bins=50)
plt.show()


In [ ]:
"new metrics"
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, accuracy_score, r2_score

def propagate_knn(Z_latent, y_true, missing_frac=0.2, k=15, tau=1.0):
    """Perform kNN label propagation on latent vectors."""
    num_rows = Z_latent.shape[0]
    dist_matrix = np.zeros((num_rows, num_rows))
    
    # compute L2 distances
    for i in range(num_rows):
        u = Z_latent[i]
        for j in range(i, num_rows):
            v = Z_latent[j]
            d = np.linalg.norm(u - v)
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d

    # mask random fraction
    num_missing = int(missing_frac * num_rows)
    missing_idx = np.random.choice(num_rows, size=num_missing, replace=False)
    y_input     = y_true.copy().astype(float)
    y_input[missing_idx] = np.nan

    # kNN neighbors
    neighbors_idx = np.argsort(dist_matrix, axis=1)[:, 1:k+1]

    # edge weights
    def compute_weights(dists):
        w = np.exp(-dists / tau)
        return w / w.sum()

    # propagate labels
    y_prop = y_input.copy()
    for i in range(num_rows):
        if np.isnan(y_input[i]):
            neigh     = neighbors_idx[i]
            neigh_y   = y_input[neigh]
            mask      = ~np.isnan(neigh_y)
            if np.sum(mask) == 0:
                continue
            neigh_y   = neigh_y[mask]
            neigh_d   = dist_matrix[i, neigh][mask]
            w         = compute_weights(neigh_d)
            y_prop[i] = np.sum(w * neigh_y)

    return y_prop, missing_idx

def evaluate_metrics(y_true, y_pred, missing_idx):
    """Compute RMSE, MAE, Accuracy, Spearman correlation, and R² for masked points."""
    y_true_masked = y_true[missing_idx]
    y_pred_masked = y_pred[missing_idx]
    mask_valid    = ~np.isnan(y_pred_masked)
    
    if mask_valid.sum() == 0:
        return {k: np.nan for k in ["rmse","mae","acc","spearman","r2"]}
    
    y_true_valid = y_true_masked[mask_valid]
    y_pred_valid = y_pred_masked[mask_valid]


    def safe_spearmanr(x, y):
        if np.std(x) == 0 or np.std(y) == 0:
            return np.nan
        return spearmanr(x, y).correlation

    metrics = {
        "rmse": np.sqrt(mean_squared_error(y_true_valid, y_pred_valid)),
        "mae": mean_absolute_error(y_true_valid, y_pred_valid),
        "acc": accuracy_score(y_true_valid, np.round(y_pred_valid).astype(int)),
        "spearman": safe_spearmanr(y_true_valid, y_pred_valid),
        "r2": r2_score(y_true_valid, y_pred_valid)
    }
    return metrics

def format_metrics(metrics: dict) -> dict:
    """Format all float metrics to 3 decimal places, keep nan as-is."""
    return {k: (f"{v:.3f}" if isinstance(v, float) and not np.isnan(v) else v)
            for k, v in metrics.items()}

# --- Hybrid AE ---
y_prop_hybrid, missing_idx = propagate_knn(z_hybrid_encoded_np, df.iloc[:,1].values)
metrics_hybrid = evaluate_metrics(df.iloc[:,1].values, y_prop_hybrid, missing_idx)
print("Hybrid metrics:", format_metrics(metrics_hybrid))

# --- Full Euclidean AE ---
y_prop_euc, missing_idx = propagate_knn(z_euclid_full_np, df.iloc[:,1].values)
metrics_euc = evaluate_metrics(df.iloc[:,1].values, y_prop_euc, missing_idx)
print("Full Euclidean metrics:", format_metrics(metrics_euc))

# --- Baseline (mean predictor) ---
y_true      = df.iloc[:,1].values
num_rows    = len(y_true)
num_missing = int(0.2 * num_rows)
np.random.seed(42)
missing_idx = np.random.choice(num_rows, size=num_missing, replace=False)

y_baseline = y_true.copy().astype(float)
y_baseline[missing_idx] = np.nan
mean_label = np.nanmean(y_baseline)
y_baseline[np.isnan(y_baseline)] = mean_label

metrics_baseline = evaluate_metrics(y_true, y_baseline, missing_idx)
print("Baseline metrics:", format_metrics(metrics_baseline))
